# 02c — Medication features

Identify qualifying recent antibiotic exposure. One row is retained per hospital spell.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell

# Define data path
DATA_DIR = get_path('data/synthetic/10000p') # Manually defined path
DATA_DIR = get_data_root_from_schema()       # Paramter schema_path= / default_dir=      
DATA_PROC_DIR = DATA_DIR / 'processed' 
KEYS = ["SUBJECT", "SPELL_IDENTIFIER"]
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


In [ ]:
# --- OPTION A: Load using pandas ---
def load_from_pandas(DATA_DIR):
    df_episodes = pd.read_parquet(DATA_DIR / "icare_episodes_anon.parquet")
    df_episodes = keep_first_spell(df_episodes)
    df_episodes["ADMISSION_DATE"] = pd.to_datetime(df_episodes["ADMISSION_DATE"])
    df_episodes["DISCHARGE_DATE"] = pd.to_datetime(df_episodes["DISCHARGE_DATE"])

    df_administration = pd.read_csv(DATA_DIR / "icare_pharmacy_administration_anon.csv")

    return {
        'episodes': df_episodes,
        'administration': df_administration
    }

# Load
dfs = load_from_pandas(DATA_DIR)
df_episodes = dfs['episodes']
df_administration = dfs['administration']

episodes: 10,000 rows -> 10,000 unique spells


In [7]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
schema_cfg = 'configs/schema.yaml'
phenotype_cfg = 'configs/phenotypes.yaml'
domains = ['episodes', 'administration']

# Load
dfs = load_from_duckdb(
    schema_path=schema_cfg,
    phenotype_path=phenotype_cfg,
    domains_to_load=domains,
    custom_codes={}, skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = keep_first_spell(dfs['episodes'])

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Loaded... episodes (2.46MB) shape (10000, 11)
Loaded... administration (104.10MB) shape (176260, 13) | Filtered codes: None
episodes: 10,000 rows -> 10,000 unique spells


In [8]:
# Libraries
from icare_risk.clinphen.utils import get_qualifying_last_abx

target_abx = [
    "co-amoxiclav", "ciprofloxacin", "ceftriaxone",
    "ertapenem", "meropenem",
]

df_medications = get_qualifying_last_abx(
    df_episodes=df_episodes,
    df_admin=df_administration,
    target_abx=target_abx,
    subject_col="SUBJECT",
    min_duration_hours=24,
    verbose=True,
)
df_medications = keep_first_spell(df_medications, "medication features")
display(df_medications.head())


--- Qualifying Last Antibiotic Summary (> 24h) ---
  Qualifying ABX Events :       0 / 10,000 (0.0%)

  No qualifying antibiotics identified.
--------------------------------------------------
medication features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,qualifying_last_abx
0,17469,6137058,7814119,3577184,2020-01-01,01:22:00,78,2020-01-27,301,gastroenterology,10,NaN
1,17849,4762859,6599332,3922801,2020-01-01,01:49:00,92,2020-01-23,300,general medicine,none,NaN
2,11176,2025106,5157422,6392675,2020-01-01,02:02:00,90,2020-01-26,140,oral surgery,7,NaN
3,18719,7291644,1727108,6718167,2020-01-01,08:53:00,43,2020-01-25,320,cardiology,7,NaN
4,16945,6678875,2124706,2441240,2020-01-01,11:58:00,83,2020-01-23,300,general medicine,2,NaN


In [9]:
filename = "02d_derived_features_medications.parquet"
output_path = DATA_PROC_DIR / filename
df_medications.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/data/synthetic/10000p/processed/02d_derived_features_medications.parquet
